<a href="https://colab.research.google.com/github/TakudzwanasheChiunzi/Taku-ISYS2001-Lab-Exit-Tickets/blob/main/starter_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

THE SIX STEP METHODOLOGY PROCESS

In [2]:
!pip install gradio

In [3]:
!pip install gradio pandas hands-on-ai
import os

# Securely bind the university infrastructure parameters directly as strings
os.environ['HANDS_ON_AI_SERVER'] = 'https://ollama.serveur.au'
os.environ['HANDS_ON_AI_MODEL'] = 'llama3.2'
# Hardcoded to bypass getpass hung-task timeouts cleanly
os.environ['HANDS_ON_AI_API_KEY'] = 'isys2001-assignment-key'

print("✅ Server Environment Parameters Locked and Loaded!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 23.3 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.14.0
    Uninstalling jiter-0.14.0:
      Successfully uninstalled jiter-0.14.0
✅ Server Environment Parameters Locked and Loaded!


In [4]:
from hands_on_ai.chat import get_response

try:
    # Testing the precise function verified in your class handout
    test_response = get_response("Hello! This is a connection handshake test.")
    print("🚀 Hands-on-AI connection successful!")
    print(f"Response from campus server: {test_response}")
except Exception as e:
    print(f"❌ Connection issue still persisting: {e}")

🧠 Loading model 'llama3.2' into RAM... give me a sec...


[WARNING] Error during request (attempt 1): Connection error.


Hang tight, I'm thinking... trying again!


[WARNING] Error during request (attempt 2): Connection error.


🚀 Hands-on-AI connection successful!
Response from campus server: ❌ Error: Connection error.


In [ ]:
import os
import gradio as gr
import pandas as pd
from datetime import datetime
import calendar

# Import the exact package frameworks from your university starter notebook
try:
    from hands_on_ai.chat import get_response
    from hands_on_ai.rag import create_index
    from hands_on_ai.agents import create_agent
except ImportError:
    pass

# Global handle to maintain the live RAG data reference across functions
budget_rag_index = None

# =====================================================================
# 🤖 AGENT TOOL MODULE (Deterministic Calculation Engine)
# =====================================================================

def calculate_budget_runway(available_cash, total_spend_cap, active_expenses_total):
    """
    Agent Tool: Computes financial runway, temporal metrics, and 80/20 compliance.
    Returns a clean, clear text statement for the Agent or user display UI.
    """
    try:
        cash = float(available_cash) if available_cash else 0.0
        cap = float(total_spend_cap) if total_spend_cap else 0.0
        spent = float(active_expenses_total) if active_expenses_total else 0.0

        today = datetime.now().date()
        current_day = today.day
        _, total_days_in_month = calendar.monthrange(today.year, today.month)
        days_remaining = (total_days_in_month - current_day) + 1

        net_balance = cash - spent
        safe_daily_allowance = (net_balance / days_remaining) if (days_remaining > 0 and net_balance > 0) else 0.0
        target_savings_reserve = cash * 0.20
        savings_rule_achieved = net_balance >= target_savings_reserve

        report = (
            f"--- Core Budget Calculations ---\n"
            f"- Total Cash Remaining: ${net_balance:,.2f}\n"
            f"- Days Remaining in Month: {days_remaining} day(s)\n"
            f"- Safe Daily Allowance: ${safe_daily_allowance:,.2f} per day\n"
            f"- Target Cap Status: " + (f"Under control (Spent ${spent:,.2f} out of a ${cap:,.2f} limit)." if spent <= cap else f"Over limit by ${(spent - cap):,.2f}!") + "\n"
            f"- 80/20 Strategy Check: " + (f"Success! Retained over 20% savings (${target_savings_reserve:,.2f})." if savings_rule_achieved else f"Shortfall! Missed the 20% target by ${(target_savings_reserve - net_balance):,.2f}.")
        )
        return report
    except Exception as e:
        return f"Error executing calculation tool metrics: {str(e)}"

# =====================================================================
# 📝 FILE DATA CLEANING & NATIVE RAG INDEX GENERATION
# =====================================================================

def load_and_index_budget_csv(file_obj):
    global budget_rag_index

    if file_obj is None:
        return None, "⚠️ No statement file detected. Please upload a valid CSV bank statement."

    try:
        df = pd.read_csv(file_obj.name)
        df.columns = [col.strip() for col in df.columns]

        if 'Amount' not in df.columns:
            return None, "⚠️ Validation Failed: File must contain an 'Amount' column header."

        df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
        df = df.dropna(subset=['Amount'])
        df = df[df['Amount'] > 0]

        clean_path = "processed_budget_ledger.csv"
        df.to_csv(clean_path, index=False)

        try:
            budget_rag_index = create_index(clean_path)
            status_msg = f"✅ RAG Data Index Compiled Successfully!"
        except Exception:
            budget_rag_index = "local_fallback_mode"
            status_msg = f"⚠️ University Server Down. System running tracking locally!"

        return df, status_msg

    except Exception as e:
        return None, f"⚠️ Error initializing data structure files: {str(e)}"

# =====================================================================
# 💬 CHAT GENERATOR CONTAINER (RAG Document Index + Agent Routing)
# =====================================================================

def process_budget_chat(user_message, chat_history, cash_input, cap_input, active_df):
    global budget_rag_index
    if not user_message:
        return "", chat_history

    try:
        total_tracked_spent = float(active_df['Amount'].sum()) if active_df is not None and not active_df.empty else 0.0
    except Exception:
        total_tracked_spent = 0.0

    system_prompt = f"""You are 'Budget Buddy', a friendly, conversational financial assistant.
    You have direct permission to use a tool called 'calculate_budget_runway' to get real-time metrics,
    and you have an index containing the user's specific text bank transactions.
    """

    try:
        tool_dictionary = {
            "calculate_budget_runway": lambda: calculate_budget_runway(cash_input, cap_input, total_tracked_spent)
        }
        agent = create_agent(tools=tool_dictionary, system=system_prompt)
        ai_reply = agent.run(user_message, index=budget_rag_index)

    except Exception:
        calc_summary = calculate_budget_runway(cash_input, cap_input, total_tracked_spent)
        ai_reply = f"☕ **[Local Fallback Mode Active]** Server connection limited. Here are your system calculations:\n\n{calc_summary}"

    chat_history.append({"role": "user", "content": user_message})
    chat_history.append({"role": "assistant", "content": str(ai_reply)})
    return "", chat_history

# =====================================================================
# 🎨 RE-RENDER ENGINE (Visual Theme Customizer)
# =====================================================================

def update_dashboard_vibes(color_preset):
    if color_preset == "rose (Spring Blossom Pink)":
        bg, card, btn, txt = "#fef1f3", "#fcc6d6", "#ef3167", "#4d0014"
    elif color_preset == "legolas (Organic Green)":
        bg, card, btn, txt = "#f2f7e0", "#c8da7f", "#587638", "#1a2408"
    elif color_preset == "glacier (Calming Blue)":
        bg, card, btn, txt = "#f0f4f9", "#ccdbff", "#153c67", "#031121"
    elif color_preset == "winnie the pooh (Yellow-Orange)":
        bg, card, btn, txt = "#fffdf0", "#ffd670", "#ff8c00", "#4d2b00"
    else:
        bg, card, btn, txt = "#f0e6da", "#dfab6e", "#8b4513", "#3d1e0a"

    style_html = f"""
    <style>
        .gradio-container, body, html {{ background-color: {bg} !important; color: {txt} !important; font-family: 'Inter', sans-serif !important; }}
        .cozy-banner {{ background-color: {card} !important; border-radius: 12px; padding: 24px; text-align: center; margin-bottom: 20px; }}
        .cozy-banner h1 {{ color: {txt} !important; margin: 0; }}
        button.primary {{ background-color: {btn} !important; color: white !important; border: none !important; font-weight: bold !important; border-radius: 8px !important; }}
    </style>
    """
    return style_html, f"✨ Mood layout shifted to '{color_preset}'!"

# =====================================================================
# GRADIO APPLICATION DESIGNS
# =====================================================================
with gr.Blocks(title="Budget Buddy Smart System") as app:
    style_injection = gr.HTML("")

    with gr.Row():
        color_preset = gr.Radio(
            choices=["cocoa-butter (Warm Brown)", "rose (Spring Blossom Pink)", "legolas (Organic Green)", "glacier (Calming Blue)", "winnie the pooh (Yellow-Orange)"],
            value="cocoa-butter (Warm Brown)", label="Pick Your Cozy Color Vibe"
        )
        theme_btn = gr.Button("🎨 Apply Theme Aesthetics", variant="secondary")

    gr.Markdown("---")

    gr.HTML('<div class="cozy-banner"><h1>☕ Your Personal Budget Buddy</h1><p>Harnessing native Agent Tools and RAG text indices to secure your financial objectives! 🦉</p></div>')

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🌟 Step 1: Initialize Operating Parameters")
            cash_val = gr.Number(value=2000, label="💰 Total Available Operating Cash ($)")
            cap_val = gr.Number(value=500, label="🎯 Target Maximum Spending Limit ($)")

            gr.Markdown("### 📊 Step 2: Load Bank Statement rows into RAG Database")
            csv_file = gr.File(file_types=[".csv"], label="Select bank statement export file")
            index_btn = gr.Button("🚀 Rebuild RAG Vector Database Index", variant="primary")
            system_ticker = gr.Textbox(value="📭 Index uninitialized.", label="RAG System Status updates", interactive=False)

        with gr.Column():
            gr.Markdown("### 💬 Step 3: Consult Your Budget Agent Advisor")
            chatbot_ui = gr.Chatbot(label="Budget Buddy Active Conversation Terminal", type="messages")
            msg_input = gr.Textbox(placeholder="Ask me anything: 'run tool' or 'Show my allowance'", label="Type Chat Prompt...")
            send_btn = gr.Button("💬 Dispatch Query to Agent", variant="primary")

    data_preview = gr.DataFrame(label="Active CSV Document Structure Records", value=pd.DataFrame(columns=["Amount"]))

    theme_btn.click(fn=update_dashboard_vibes, inputs=[color_preset], outputs=[style_injection, system_ticker])
    index_btn.click(fn=load_and_index_budget_csv, inputs=[csv_file], outputs=[data_preview, system_ticker])

    send_btn.click(
        fn=process_budget_chat,
        inputs=[msg_input, chatbot_ui, cash_val, cap_val, data_preview],
        outputs=[msg_input, chatbot_ui]
    )

app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4028048b76dad6ab0f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Step 1: Redefining the Problem
I chose my finance assistant to operate as a budget buddy. The primary focus is that the assistant will be able to take in csv files filled with expenses or allow the user to manually input their expenses. The app will use this information and calculate the

## Step 2: Inputs and Outputs

Inputs:

-	Expenses
-	Budget_Amount
-	Available_Money

Output:

-	Percentage_of_budget_used
-	Percentage_of_available_money_used
-	Net Balance
-	Total_Expenses
-	Chatbot Advice

UPDATED INPUTS AND OUTPUTS:

Inputs:

- Expenses: the value of the expenses (integer) | must be greater than 0
- Budget_Amount: how much they want to spend (integer) | must be greater than 0
- Available_Money: how much money they have overall (integer) | must be greater than 0
- CSV file: a file of their expenses if they don’t want to input each expense (integer)
- Expense_Name: the meaning of the expenses (string)
- Date: the date of the expense
- Scheduled_Pay_Day: when are the next expecting to receive money
- Category: if the expense is an essential/need or non-essential/want


Outputs:

- Percentage_of_budget_used ( also calling it Burn_Rate)
- Percentage_of_available_money_used (also calling it Drawdown)
- Net Balance
- Total_Expenses
- Chatbot Advice
- Runaway : the days left before they get more money


## Step 3: Calculations

- Percentage of budget used:
= (Total_Expenses/Budget_Amount) * 100

- Percentage of available money used:
= Total_Expenses/ Available_Money * 100

- Total Expenses:
= (Sum of Expenses)

- Net Balance
= Available_Money – Total_Expenses

UPDATED CALCULATIONS:

- Percentage of budget used: = (Total_Expenses/Budget_Amount) * 100
- Percentage of available money used: = Total_Expenses/ Available_Money * 100
- Total Expenses: = (Sum of Expenses)
- Net Balance = Available_Money – Total_Expenses


## Step 4: Pseudo Code

•	The app will welcome the user to the app

•	The app will ask the user to select a theme and font

•	App will start by asking if the user wants to see the menu/dashboard or jump into the budget

If the User chooses Budget:

•	App asks if the user wants to upload a CSV file or input transactions If the User chooses CSV:

•	App prompts user to upload the CSV

•	App calculates total spent, amount of budget used, percentage of budget used, and gives out advice If the User chooses input transactions

•	App prompts user to input how many transactions they want to input

•	App asks user to input the name of the transactions, what it was for and the amount

•	App calculates total spent, amount of budget used, percentage of budget used and gives out advice

If the user chooses dashboard/menu: Dashboard will show:

•	Expenses

•	Chatbot (user can ask budget-related advice and questions)


UPDATED PSEUDOCODE:

Inputs:
- Expenses: the value of the expenses (integer) | must be greater than 0
- Budget_Amount: how much they want to spend (integer) | must be greater than 0
- Available_Money: how much money they have overall (integer) | must be greater than 0
- Expense_Name: the meaning of the expenses (string)
- Date: the date of the expense
- Scheduled_Pay_Day: when are the next expecting to receive money
- Category: if the expense is an essential/need or non-essential/want
- CSV file: a file of their expenses if they don’t want to input each expense (integer)

Outputs:

- Burn_Rate (the percentage of budget used)
- Drawdown (the percentage of available money used)
- Net Balance
- Total_Expenses
- Chatbot Advice
- Runaway : the days left before they get more money

Step 3: Calculations
- Burn_Rate: = (Total_Expenses/Budget_Amount) * 100 (if >budget then user overspent, if <budget then user successfully went below budget which is good, if = budget, user spent according to plan)
- Drawdown: = Total_Expenses/ Available_Money * 100 (if available money < total expenses and <budget user needs financial advice)
- Total Expenses: = (Sum of Expenses)
- Net Balance = Available_Money – Total_Expenses

Key things for the code:
- Burn_Rate:

if  Burn_Rate > budget then user overspent

if Burn_Rate < budget then user successfully went below budget which is good

if  Burn_Rate = budget user spent according to plan

- Drawdown:

if Drawdown < total expenses and Drawdown < budget user needs financial advice

2ND UPDATE TO PSEUDOCODE

•	The app will welcome the user to the app

•	The app will ask the user to select a theme and font

•	App will start by asking if the user wants to see the menu/dashboard or jump into the budget

If the User chooses Budget:

•	App asks if the user wants to upload a CSV file or input transactions

If the User chooses CSV:

•	App prompts user to upload the CSV

•	App calculates the above mentioned calculations and gives advice

If the User chooses input transactions:

•	App prompts user to input transactions and type the word “done” when done adding transactions

•	App calculates the above mentioned calculations and gives out advice

If the user chooses dashboard/menu: Dashboard will show:

•	Budget tab (where user can click it and view their expenses or start the budget tracking process as described above)

•	Chatbot (user can ask budget-related advice and questions)


GEMINI PSUEDO CODE- CHECK THE CELL BELOW

## Step 5 & Step 6: Code and Testing
Refer to code cells below.



**GEMINI'S PSEUDOCODE**

## Step 4: Write Pseudo Code (Application Execution Logic)

Below is the foundational recipe and algorithmic architecture explaining how the interactive system functions step-by-step before it is compiled into running Python commands.

### Main Control Flow Matrix

1. **INITIALIZATION & BRANDING STAGE:**
   - Initialize the Gradio block application UI.
   - Set visual design elements: Inject custom typography fonts and apply the "Soft" sleek corporate layout theme template.
   - Render text display headers: Present a clear user welcome banner introducing "Budget Buddy" as an analytical optimization framework.

2. **DASHBOARD NAVIGATION MATRIX:**
   - Generate two functional workspace directories using structural Tab Layouts:
     - **Tab A:** "Budget Analytics Workspace"
     - **Tab B:** "Financial Advisory Chatbot"

3. **TAB A PROCESSING LOGIC (Financial Calculations):**
   - Prompt user to input two mandatory scalar numeric criteria: `Available_Money` and `Budget_Amount`.
   - Present a data ingestion branch options toggle selector (Radio inputs):
     
     * **IF User Selects "Upload CSV File":**
       - Render a file attachment upload upload box constraint.
       - Await CSV file parsing via Pandas libraries.
       - Read data rows, target the transaction value arrays, and compute the total sum aggregate value.
       - Define: `Total_Expenses = Sum of CSV values`.
       
     * **IF User Selects "Input Transactions Manually":**
       - Open an interactive transactional array list logger container.
       - LOOP: Continually capture user inputs for specific transaction item labels and currency quantities.
       - CONSTRAINT BREAKPOINT: Detect string matches. If the user submits the exact terminal command phrase "done", HALT input collection loop.
       - Process loop results: Aggregate all compiled item records together.
       - Define: `Total_Expenses = Sum of gathered manual quantities`.

   - **EXECUTE BACKEND FINANCIAL FORMULAS:**
     - Check constraint: IF `Budget_Amount` equals 0, output system alert flag "Budget Cap cannot be zero" to avoid terminal ZeroDivisionError system crashes.
     - Calculate: `Net Balance = Available_Money - Total_Expenses`
     - Calculate: `Monthly Budget Burn Rate = (Total_Expenses / Budget_Amount) * 100`
     - Calculate: `Liquid Capital Drawdown = (Total_Expenses / Available_Money) * 100`

   - **OUTPUT DISPLAY GENERATION:**
     - Render mathematical answers clearly inside designated text summary fields.
     - Append conditional data summaries behind the scenes. Route these calculated values into the AI prompt window as contextual reference briefings so the bot can craft highly accurate, personalized advice automatically.

4. **TAB B PROCESSING LOGIC (Chatbot Interaction Engine):**
   - Display a scrollable interactive dialog text stream log.
   - Instantly pass user query inquiries directly to the registered `hands-on-ai` chatbot package engine wrapper.
   - Command the AI model to consistently maintain a professional, corporate financial coach identity.
   - Output responses clearly back to the user view tab interface window.